# Unity Catalog Rebuild And Documentation Refresh

Single local workflow for rebuilding Ampere Unity Catalog metadata from the canonical contract and refreshing generated documentation artifacts.

Pipeline steps:
1. Regenerate DAG config from `tools/uc/contracts/ampere_tables.json`.
2. Recreate or repair UC metadata for selected layers.
3. Query UC back for layer-level table counts.
4. Extract compact per-layer contracts into `docs/data_contracts/`.
5. Regenerate generated dataflow documentation under `docs/dataflow/`.


In [1]:
# -----------------------------------------------------------------------------
# Manual setup
# -----------------------------------------------------------------------------
LAYERS = ["bronze", "silver", "gold"]
CONTRACT_PATH = "tools/uc/contracts/ampere_tables.json"
OUTPUT_DIR = "docs/data_contracts"
REPORT_DIR = "tools/uc/reports"

REGENERATE_DAG_CONFIG = True
REBUILD_UC_CATALOG = True
CHECK_UC_CATALOG = True
EXTRACT_DOCS_CONTRACTS = True
REGENERATE_DOCS = True
VERBOSE = True


In [2]:
# -----------------------------------------------------------------------------
# Step 0: import reusable primitives and initialize result tracking
# -----------------------------------------------------------------------------
from datetime import datetime, timezone
from pathlib import Path
import json
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find project root from current notebook path.")

from tools.docs import generate_dataflow_docs  # noqa: E402
from tools.py_utils.paths import project_root  # noqa: E402
from tools.py_utils.uc_client import UCClient  # noqa: E402
from tools.py_utils.uc_config import catalog_name, load_uc_bulk_config, uc_client_config  # noqa: E402
from tools.py_utils.uc_contracts import (  # noqa: E402
    layer_schema_names,
    layer_schemas,
    layer_uc_payload,
    layers,
    load_contract,
)
from tools.py_utils.uc_registration import (  # noqa: E402
    empty_registration_report,
    ensure_catalog,
    ensure_schemas,
    load_table_contracts,
    replace_external_tables,
    write_report,
)
from tools.uc.generators import generate_from_contract  # noqa: E402

root = project_root(__file__ if "__file__" in globals() else PROJECT_ROOT)
selected_layers = list(LAYERS)
results = {
    "project_root": str(root),
    "layers": selected_layers,
    "dag_config_regenerated": False,
    "uc_reports": [],
    "uc_table_counts": {},
    "data_contracts": [],
    "docs_regenerated": False,
}
results


{'project_root': 'C:\\Users\\likelol\\Projects\\ampere_project',
 'layers': ['bronze', 'silver', 'gold'],
 'dag_config_regenerated': False,
 'uc_reports': [],
 'uc_table_counts': {},
 'data_contracts': [],
 'docs_regenerated': False}

In [3]:
# -----------------------------------------------------------------------------
# Shared notebook-local functions for UC rebuild and docs extraction
# -----------------------------------------------------------------------------
def create_uc_client(verbose=True):
    config = load_uc_bulk_config()
    return UCClient(uc_client_config(config, verbose=verbose)), config


def prepare_layer(layer, contract_path=CONTRACT_PATH, report_dir=REPORT_DIR, verbose=True):
    contract = load_contract(contract_path)
    contract_layers = layers(contract)
    if layer not in contract_layers:
        raise ValueError(f"Unknown layer {layer!r}. Available: {sorted(contract_layers)}")

    uc, config = create_uc_client(verbose=verbose)
    report = empty_registration_report(
        catalog=uc.config.catalog,
        layer_name=layer,
        report_format=contract_layers[layer].get("format", "DELTA").upper(),
    )
    ensure_catalog(uc, comment=f"Ampere {uc.config.catalog} catalog", report=report)
    ensure_schemas(uc, layer_schemas(contract, layer), report=report)

    payload_path = root / report_dir / f"uc_{layer}_input.json"
    payload_path.parent.mkdir(parents=True, exist_ok=True)
    payload_path.write_text(
        json.dumps(layer_uc_payload(contract, layer), indent=2) + "\n",
        encoding="utf-8",
    )
    specs = load_table_contracts(payload_path)
    replace_external_tables(uc, specs, report)

    report_path = root / report_dir / f"uc_{layer}_report.json"
    report["config_catalog"] = config["catalog"]
    write_report(report, report_path)
    return report_path


def check_layer(layer, contract_path=CONTRACT_PATH, verbose=True):
    contract = load_contract(contract_path)
    contract_layers = layers(contract)
    if layer not in contract_layers:
        raise ValueError(f"Unknown layer {layer!r}. Available: {sorted(contract_layers)}")

    uc, _ = create_uc_client(verbose=verbose)
    tables = []
    for schema in layer_schemas(contract, layer):
        schema_name = schema["name"]
        registered = uc.list_tables(schema_name)
        print(f"Tables in {uc.config.catalog}.{schema_name}: {len(registered)}")
        tables.extend(registered)
    return tables


def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def compact_column(column):
    return {
        "name": column.get("name"),
        "type_name": column.get("type_name"),
        "type_text": column.get("type_text"),
        "position": column.get("position"),
        "nullable": column.get("nullable"),
        "comment": column.get("comment"),
    }


def compact_table(table, layer, catalog):
    columns = sorted(table.get("columns") or [], key=lambda item: int(item.get("position") or 0))
    return {
        "name": table.get("name") or table.get("table_name"),
        "catalog": table.get("catalog_name") or catalog,
        "schema": table.get("schema_name") or layer,
        "table_type": table.get("table_type"),
        "data_source_format": table.get("data_source_format"),
        "storage_location": table.get("storage_location"),
        "comment": table.get("comment"),
        "columns": [compact_column(column) for column in columns],
    }


def extract_data_contracts(layers_to_extract=None, output_dir=OUTPUT_DIR, verbose=True):
    contract = load_contract()
    config = load_uc_bulk_config()
    catalog = catalog_name(config)
    uc, _ = create_uc_client(verbose=verbose)
    output_path = root / output_dir
    output_path.mkdir(parents=True, exist_ok=True)
    extracted_at_utc = utc_now_iso()
    written = []

    for layer in layers_to_extract or ["bronze", "silver", "gold"]:
        layer_tables = []
        for schema_name in layer_schema_names(contract, layer):
            layer_tables.extend(uc.list_tables(schema_name))
        payload = {
            "layer": layer,
            "catalog": catalog,
            "schema": layer,
            "source": uc.base_url,
            "extracted_at_utc": extracted_at_utc,
            "tables": sorted(
                [compact_table(table, layer, catalog) for table in layer_tables],
                key=lambda item: item["name"] or "",
            ),
        }
        path = output_path / f"{layer}.json"
        path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        print(f"Wrote {path}: tables={len(payload['tables'])}")
        written.append(path)
    return written


In [4]:
# -----------------------------------------------------------------------------
# Step 1: regenerate derived DAG config from the canonical contract
# -----------------------------------------------------------------------------
if REGENERATE_DAG_CONFIG:
    print("Regenerating DAG config from canonical contract...")
    generate_from_contract.main()
    results["dag_config_regenerated"] = True
else:
    print("Skipped DAG config regeneration.")

results["dag_config_regenerated"]


Regenerating DAG config from canonical contract...
Generated artifacts from tools/uc/contracts/ampere_tables.json


True

In [5]:
# -----------------------------------------------------------------------------
# Step 2: rebuild Unity Catalog metadata for selected layers
# -----------------------------------------------------------------------------
if REBUILD_UC_CATALOG:
    for layer in selected_layers:
        print(f"Preparing UC metadata for layer: {layer}")
        report_path = prepare_layer(layer=layer, contract_path=CONTRACT_PATH, verbose=VERBOSE)
        results["uc_reports"].append(str(report_path))
else:
    print("Skipped UC metadata rebuild.")

results["uc_reports"]


Preparing UC metadata for layer: bronze
[debug] remote cmd: kubectl -n unity-catalog get endpoints unity-catalog-unitycatalog-server --no-headers | awk '{print $2}' | cut -d: -f1
[debug] stdout: 10.10.4.52
[debug] remote cmd: curl -sS --connect-timeout 10 --max-time 60 -X GET http://10.10.4.52:8080/api/2.1/unity-catalog/catalogs
[debug] stdout: {"catalogs":[{"name":"ampere","comment":null,"properties":{},"owner":null,"created_at":1771755058153,"created_by":null,"updated_at":1771755058153,"updated_by":null,"id":"121105d5-19cd-48a4-b370-baec6a411b3a","storage_root":null,"storage_location":null}],"next_page_token":null}
[catalog] exists: ampere
[debug] remote cmd: curl -sS --connect-timeout 10 --max-time 60 -X GET http://10.10.4.52:8080/api/2.1/unity-catalog/schemas -G --data-urlencode catalog_name=ampere
[debug] stdout: {"schemas":[{"name":"bronze","catalog_name":"ampere","comment":null,"properties":{},"full_name":"ampere.bronze","owner":null,"created_at":1771755097828,"created_by":null,

['C:\\Users\\likelol\\Projects\\ampere_project\\tools\\uc\\reports\\uc_bronze_report.json',
 'C:\\Users\\likelol\\Projects\\ampere_project\\tools\\uc\\reports\\uc_silver_report.json',
 'C:\\Users\\likelol\\Projects\\ampere_project\\tools\\uc\\reports\\uc_gold_report.json']

In [6]:
# -----------------------------------------------------------------------------
# Step 3: read Unity Catalog metadata back for a quick verification pass
# -----------------------------------------------------------------------------
if CHECK_UC_CATALOG:
    for layer in selected_layers:
        print(f"Checking UC metadata for layer: {layer}")
        tables = check_layer(layer=layer, contract_path=CONTRACT_PATH, verbose=VERBOSE)
        results["uc_table_counts"][layer] = len(tables)
else:
    print("Skipped UC metadata check.")

results["uc_table_counts"]


Checking UC metadata for layer: bronze
[debug] remote cmd: kubectl -n unity-catalog get endpoints unity-catalog-unitycatalog-server --no-headers | awk '{print $2}' | cut -d: -f1
[debug] stdout: 10.10.4.52
[debug] remote cmd: curl -sS --connect-timeout 10 --max-time 60 -X GET http://10.10.4.52:8080/api/2.1/unity-catalog/tables -G --data-urlencode catalog_name=ampere --data-urlencode schema_name=bronze
[debug] stdout: {"tables":[{"name":"assortment","catalog_name":"ampere","schema_name":"bronze","table_type":"EXTERNAL","data_source_format":"DELTA","columns":[{"name":"product_id","type_text":"smallint","type_json":"{\"name\": \"short\"}","type_name":"SHORT","type_precision":null,"type_scale":null,"type_interval_type":null,"position":1,"comment":null,"nullable":true,"partition_index":null},{"name":"store_id","type_text":"smallint","type_json":"{\"name\": \"short\"}","type_name":"SHORT","type_precision":null,"type_scale":null,"type_interval_type":null,"position":2,"comment":null,"nullable":

{'bronze': 17, 'silver': 17, 'gold': 9}

In [7]:
# -----------------------------------------------------------------------------
# Step 4: extract documentation data contracts from live Unity Catalog state
# -----------------------------------------------------------------------------
if EXTRACT_DOCS_CONTRACTS:
    print("Extracting documentation data contracts from UC...")
    written = extract_data_contracts(layers_to_extract=selected_layers, output_dir=OUTPUT_DIR, verbose=VERBOSE)
    results["data_contracts"] = [str(path) for path in written]
else:
    print("Skipped documentation data contract extraction.")

results["data_contracts"]


Extracting documentation data contracts from UC...
[debug] remote cmd: kubectl -n unity-catalog get endpoints unity-catalog-unitycatalog-server --no-headers | awk '{print $2}' | cut -d: -f1
[debug] stdout: 10.10.4.52
[debug] remote cmd: curl -sS --connect-timeout 10 --max-time 60 -X GET http://10.10.4.52:8080/api/2.1/unity-catalog/tables -G --data-urlencode catalog_name=ampere --data-urlencode schema_name=bronze
[debug] stdout: {"tables":[{"name":"assortment","catalog_name":"ampere","schema_name":"bronze","table_type":"EXTERNAL","data_source_format":"DELTA","columns":[{"name":"product_id","type_text":"smallint","type_json":"{\"name\": \"short\"}","type_name":"SHORT","type_precision":null,"type_scale":null,"type_interval_type":null,"position":1,"comment":null,"nullable":true,"partition_index":null},{"name":"store_id","type_text":"smallint","type_json":"{\"name\": \"short\"}","type_name":"SHORT","type_precision":null,"type_scale":null,"type_interval_type":null,"position":2,"comment":null

['C:\\Users\\likelol\\Projects\\ampere_project\\docs\\data_contracts\\bronze.json',
 'C:\\Users\\likelol\\Projects\\ampere_project\\docs\\data_contracts\\silver.json',
 'C:\\Users\\likelol\\Projects\\ampere_project\\docs\\data_contracts\\gold.json']

In [8]:
# -----------------------------------------------------------------------------
# Step 5: regenerate repository dataflow documentation
# -----------------------------------------------------------------------------
if REGENERATE_DOCS:
    print("Regenerating dataflow documentation...")
    generate_dataflow_docs.main()
    results["docs_regenerated"] = True
else:
    print("Skipped dataflow documentation regeneration.")

results["docs_regenerated"]


Regenerating dataflow documentation...


True

In [9]:
# -----------------------------------------------------------------------------
# Final summary
# -----------------------------------------------------------------------------
results


{'project_root': 'C:\\Users\\likelol\\Projects\\ampere_project',
 'layers': ['bronze', 'silver', 'gold'],
 'dag_config_regenerated': True,
 'uc_reports': ['C:\\Users\\likelol\\Projects\\ampere_project\\tools\\uc\\reports\\uc_bronze_report.json',
  'C:\\Users\\likelol\\Projects\\ampere_project\\tools\\uc\\reports\\uc_silver_report.json',
  'C:\\Users\\likelol\\Projects\\ampere_project\\tools\\uc\\reports\\uc_gold_report.json'],
 'uc_table_counts': {'bronze': 17, 'silver': 17, 'gold': 9},
 'data_contracts': ['C:\\Users\\likelol\\Projects\\ampere_project\\docs\\data_contracts\\bronze.json',
  'C:\\Users\\likelol\\Projects\\ampere_project\\docs\\data_contracts\\silver.json',
  'C:\\Users\\likelol\\Projects\\ampere_project\\docs\\data_contracts\\gold.json'],
 'docs_regenerated': True}